# Topic: SQL: Sessionization (Clickstream Grouping)

## Definition (30-second explanation)
* Sessionization is the process of grouping raw, sequential user events (like page views or clicks) into distinct "sessions" or visits.
* A new session typically begins when a user is inactive for a predefined time threshold (most commonly 30 minutes).

## Why Interviewers Ask This
* **Window Function Mastery:** It is the ultimate test of your ability to chain advanced window functions (`LAG` and running `SUM`).
* **Data Prep Reality:** Product data scientists rarely get perfectly formatted `session_id` columns; they usually have to generate them from raw clickstreams.
* **Business Logic Translation:** Tests your ability to convert a fuzzy business rule ("a user left the site") into strict SQL logic (time gaps).

## Core Concepts
* **The `LAG()` Peek:** Using `LAG(event_time)` to look at the timestamp of the immediately preceding event for that specific user.
* **The Boundary Flag:** Using a `CASE WHEN` statement to evaluate the time difference. If the gap > 30 minutes, flag it as `1` (new session), else `0`.
* **The Cumulative Counter:** Applying a running `SUM()` over the boundary flags. Because `0`s don't add to the sum, all events in the same session get the same cumulative ID.

## When to Use
* Calculating bounce rates, average time-on-site, or pages-per-session.
* Building user journey/path analysis (e.g., what is the most common exit page?).
* Processing raw logs from web tracking tools (like Snowplow or raw Google Analytics dumps).

## Advantages
* Transforms unstructured, infinite event streams into bounded, analyzable behavioral blocks.
* Highly customizable (e.g., a video streaming site might use a 2-hour gap, while a banking app uses a 5-minute gap).

## Limitations
* Highly computationally expensive on massive, billion-row event logs due to multiple window function passes and sorting requirements.
* Time-based heuristics aren't perfect (e.g., a user leaving a tab open while eating lunch might falsely trigger a new session upon returning).

## Common Comparisons
* **Time-based vs. Event-based Sessionization:** Time-based ends sessions after a duration of inactivity (30 mins). Event-based ends sessions upon a specific action (e.g., clicking 'Logout' or 'Complete Purchase').
* **`LAG()` vs. `LEAD()`:** `LAG()` looks backwards to find the gap from the previous event (standard for session starts). `LEAD()` looks forward to find the gap to the next event (useful for calculating time spent on the *current* page).

## Common Interview Traps
* **Missing the Partition:** Writing `LAG(event_time) OVER (ORDER BY event_time)`. This compares User B's event to User A's event, completely corrupting the sessions. Always `PARTITION BY user_id`.
* **The NULL Trap:** The very first event a user ever performs has no previous event. `LAG()` returns `NULL`. If you don't handle this (`prev_event_time IS NULL`), the first session won't trigger properly.

## Python / SQL Syntax (if applicable)

    -- Step 1 & 2: Lag and Flag
    WITH lagged AS (
        SELECT user_id, event_time,
               LAG(event_time) OVER(PARTITION BY user_id ORDER BY event_time) AS prev_time
        FROM events
    ),
    flagged AS (
        SELECT *,
               CASE WHEN prev_time IS NULL THEN 1
                    WHEN TIMESTAMPDIFF(MINUTE, prev_time, event_time) > 30 THEN 1 
                    ELSE 0 END AS is_new_session
        FROM lagged
    )
    -- Step 3: Running Sum for Session ID
    SELECT user_id, event_time,
           SUM(is_new_session) OVER(PARTITION BY user_id ORDER BY event_time) AS session_id
    FROM flagged;

## 45-Second Interview Answer
"Sessionization groups raw clickstreams into meaningful visits, usually defined by a 30-minute inactivity gap. In SQL, this is a three-step CTE process. First, I use `LAG()` partitioned by `user_id` and ordered by time to get the previous event's timestamp. Second, I calculate the time difference and use a `CASE` statement to flag a `1` if the gap exceeds 30 minutes or if it's the first event. Finally, I use a running `SUM()` over those flags to generate a monotonically increasing `session_id` for each user."

## Example Questions:

**Mock Schema and Data**
```sql
-- 1. Create the schema
CREATE TABLE user_events (
    user_id INT,
    page_url VARCHAR(255),
    event_time TIMESTAMP,
    device_type VARCHAR(50)
);

-- 2. Insert mock data
INSERT INTO user_events (user_id, page_url, event_time, device_type) VALUES
-- USER 101: 2 Sessions. Session 1 is a bounce, Session 2 converts.
(101, '/home', '2025-01-01 09:00:00', 'desktop'),
(101, '/products', '2025-01-01 09:05:00', 'desktop'),
(101, '/cart', '2025-01-01 09:10:00', 'desktop'),
-- Gap > 30 mins triggers Session 2
(101, '/home', '2025-01-01 10:15:00', 'desktop'),
(101, '/about', '2025-01-01 10:18:00', 'desktop'),
(101, '/purchase', '2025-01-01 10:20:00', 'desktop'),

-- USER 102: 2 Sessions. Both are bounces.
(102, '/home', '2025-01-01 11:00:00', 'mobile'),
-- Gap > 30 mins triggers Session 2
(102, '/blog', '2025-01-01 11:45:00', 'mobile'),
(102, '/contact', '2025-01-01 11:50:00', 'mobile'),

-- USER 103: 6 Sessions in a single day (To test the >5 sessions query)
(103, '/home', '2025-01-01 08:00:00', 'mobile'),
(103, '/search', '2025-01-01 09:00:00', 'mobile'),
(103, '/products', '2025-01-01 10:00:00', 'mobile'),
(103, '/home', '2025-01-01 11:00:00', 'mobile'),
(103, '/cart', '2025-01-01 12:00:00', 'mobile'),
(103, '/checkout', '2025-01-01 13:00:00', 'mobile');
```

**Sessionization Query**
```sql
WITH events_with_gap AS (
    SELECT 
        user_id,
        page_url,
        event_time,
        device_type,
        LAG(event_time) OVER (
            PARTITION BY user_id 
            ORDER BY event_time
        ) AS prev_event_time
    FROM user_events
),
session_flags AS (
    SELECT 
        *,
        CASE 
            WHEN prev_event_time IS NULL THEN 1
            WHEN TIMESTAMPDIFF(MINUTE, prev_event_time, event_time) > 30 THEN 1 
            ELSE 0 
        END AS is_new_session
    FROM events_with_gap
),
sessionized_events AS (
    SELECT 
        user_id,
        page_url,
        event_time,
        device_type,
        SUM(is_new_session) OVER (
            PARTITION BY user_id 
            ORDER BY event_time
        ) AS session_id
    FROM session_flags
)
-- PASTE YOUR PRACTICE QUESTION QUERIES HERE

```

### Q1. Calculate the average session duration per user
* **Ideal Interview Answer (MySQL):** *(Assuming the 3-step sessionization CTE `sessionized_events` is already built)*
```sql
    WITH session_durations AS (
        SELECT 
            user_id, 
            session_id, 
            TIMESTAMPDIFF(MINUTE, MIN(event_time), MAX(event_time)) AS duration_mins
        FROM sessionized_events
        GROUP BY user_id, session_id
    )
    SELECT 
        user_id, 
        AVG(duration_mins) AS avg_session_duration
    FROM session_durations
    GROUP BY user_id;
```
* **Common Mistakes:** Trying to average the time directly in one pass without first aggregating the MAX and MIN boundaries of the individual sessions. 
* **Likely Follow-up:** What if a session only has one event (a bounce)? What will the duration be, and how does that affect the average? *(Answer: It will be 0 minutes. If we want to exclude bounces from the average, we should add `WHERE duration_mins > 0` in the final SELECT).*

### Q2. Find users with more than 5 sessions in a single day
* **Ideal Interview Answer (MySQL):**
```sql
    SELECT 
        user_id, 
        DATE(event_time) AS session_date
    FROM sessionized_events
    GROUP BY user_id, DATE(event_time)
    HAVING COUNT(DISTINCT session_id) > 5;
```
* **Common Mistakes:** Grouping only by `user_id` and forgetting to group by the `DATE()` of the event, which would just return users with > 5 sessions over their entire lifetime.
* **Likely Follow-up:** Can a single `session_id` span across two calendar days (e.g., starting at 11:50 PM and ending at 12:15 AM)? How does your query handle that?

### Q3. Find the most visited page across all sessions
* **Ideal Interview Answer (MySQL):**
```sql
    SELECT 
        page_url, 
        COUNT(*) AS total_visits
    FROM sessionized_events
    GROUP BY page_url
    ORDER BY total_visits DESC
    LIMIT 1;
```
* **Common Mistakes:** Overcomplicating the query by factoring in the `session_id`. If the prompt just asks for the most visited page overall, you simply group by the URL and count the rows.
* **Likely Follow-up:** How would you modify this to find the most *unique* sessions a page was viewed in, rather than raw pageviews? *(Answer: Change `COUNT(*)` to `COUNT(DISTINCT CONCAT(user_id, session_id))`)*.

### Q4. Identify sessions that ended without a purchase (bounce sessions)
* **Ideal Interview Answer (MySQL):**
```sql
    SELECT 
        user_id, 
        session_id
    FROM sessionized_events
    GROUP BY user_id, session_id
    HAVING SUM(CASE WHEN page_url = '/purchase' THEN 1 ELSE 0 END) = 0;
```
* **Common Mistakes:** Using a `WHERE page_url != '/purchase'` filter before grouping. This just removes the purchase rows, but the session will still show up in the results based on its other page views. 
* **Likely Follow-up:** How would you calculate the overall "Bounce Rate" (percentage of sessions with zero purchases) from this logic?

### Q5. Build a report showing average pages per session by device type
* **Ideal Interview Answer (MySQL):** *(Assuming device_type is in the raw table)*
```sql
    WITH session_page_counts AS (
        SELECT 
            device_type,
            user_id,
            session_id,
            COUNT(*) AS pages_viewed
        FROM sessionized_events
        GROUP BY device_type, user_id, session_id
    )
    SELECT 
        device_type,
        AVG(pages_viewed) AS avg_pages_per_session
    FROM session_page_counts
    GROUP BY device_type;
```
* **Common Mistakes:** Trying to do `COUNT(*) / COUNT(DISTINCT session_id)` in one step. While mathematically possible, it's prone to logical errors if device_types somehow switch mid-session. CTEs are safer and more readable.
* **Likely Follow-up:** What could cause a session to have an artificially inflated page count? *(Answer: Bot traffic or web scrapers. We usually filter out sessions with > 100 pages/minute).*

## Practice Questions:

### Q1:
**Assume you have already successfully run the sessionization CTE and materialized it into a table called user_sessions.**

**Mock Schema and Data:**
```sql
-- 1. Create the schema
    CREATE TABLE user_sessions (
        user_id INT,
        session_id INT,
        page_url VARCHAR(255),
        event_time TIMESTAMP
    );

    -- 2. Insert mock data
    INSERT INTO user_sessions (user_id, session_id, page_url, event_time) VALUES
    -- User 101, Session 1 (Standard 3-page session)
    -- Expected Landing: '/home', Expected Exit: '/checkout'
    (101, 1, '/home', '2026-08-11 10:00:00'),
    (101, 1, '/products', '2026-08-11 10:05:00'),
    (101, 1, '/checkout', '2026-08-11 10:15:00'),

    -- User 101, Session 2 (Standard 2-page session later in the day)
    -- Expected Landing: '/blog', Expected Exit: '/about'
    (101, 2, '/blog', '2026-08-11 14:00:00'),
    (101, 2, '/about', '2026-08-11 14:10:00'),

    -- User 102, Session 1 (Bounce / Single page session)
    -- Expected Landing: '/promo', Expected Exit: '/promo'
    (102, 1, '/promo', '2026-08-11 11:00:00'),

    -- User 103, Session 1 (Multi-page with a loop back to home)
    -- Expected Landing: '/home', Expected Exit: '/home'
    (103, 1, '/home', '2026-08-11 12:00:00'),
    (103, 1, '/search', '2026-08-11 12:02:00'),
    (103, 1, '/home', '2026-08-11 12:10:00');
```

**Question: Using the user_sessions table, write a single MySQL query that returns one row per session containing the user_id, session_id, the landing_page (the first page visited in that session), and the exit_page (the last page visited in that session).**

* **Answer (MySQL):** 
```SQL
    SELECT DISTINCT 
        user_id, 
        session_id,
        FIRST_VALUE(page_url) OVER (
            PARTITION BY user_id, session_id 
            ORDER BY event_time ASC
        ) AS landing_page,
        FIRST_VALUE(page_url) OVER (
            PARTITION BY user_id, session_id 
            ORDER BY event_time DESC
        ) AS exit_page
    FROM user_sessions;
```
\*Explanation:* "Because standard aggregations like `MIN` or `MAX` don't work chronologically on text strings like URLs, I use the `FIRST_VALUE` window function. By ordering ascending by time, I get the landing page. By ordering descending, I trick the function into grabbing the last event, which acts as the exit page. Applying `DISTINCT` at the top ensures I return just one row per session."
* **Common Mistakes:** Using `PARTITION BY session_id` without including `user_id`, which mixes up sessions across different users. Attempting to use `LAST_VALUE()` without understanding that its default window frame stops at the current row, often returning the wrong page unless explicitly framed with `ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING`.
* **Likely Follow-up:** How would you modify this to find sessions where the user landed and exited on the exact same page? *(Answer: Wrap the query in a CTE and add `WHERE landing_page = exit_page` in the outer query).*